## **Ομάδα 4**

Ανθοπούλου Φαίδρα Αναστασία 03118818 

Θοδωρής-Άγγελος Μέξης 03118408


# Εργαστηριακή Άσκηση 2. Μη επιβλεπόμενη μάθηση. 
## Σύστημα συστάσεων βασισμένο στο περιεχόμενο
## Σημασιολογική απεικόνιση δεδομένων με χρήση SOM 
Ημερομηνία εκφώνησης της άσκησης: 29 Νοεμβρίου 2022

**Θα βρείτε το παρόν σε μορφή jupyter notebook ως συνημμένο στο τέλος της εκφώνησης.**


In [1]:
!pip install --upgrade pip
!pip install --upgrade numpy
!pip install --upgrade pandas
!pip install --upgrade nltk
!pip install --upgrade scikit-learn
!pip install --upgrade joblib

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Using cached numpy-1.24.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 1.24.1 which is incompatible.
numba 0.56.4 requires numpy<1.24,>=1.18, but you have numpy 1.24.1 which is incompatible.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


## Εισαγωγή του Dataset

Το σύνολο δεδομένων με το οποίο θα δουλέψουμε είναι βασισμένο στο [Carnegie Mellon Movie Summary Corpus](http://www.cs.cmu.edu/~ark/personas/). Πρόκειται για ένα dataset με 22.301 περιγραφές ταινιών. Η περιγραφή κάθε ταινίας αποτελείται από τον τίτλο της, μια ή περισσότερες ετικέτες που χαρακτηρίζουν το είδος της ταινίας και τέλος τη σύνοψη της υπόθεσής της. Αρχικά εισάγουμε το dataset (χρησιμοποιήστε αυτούσιο τον κώδικα, δεν χρειάζεστε το αρχείο csv) στο dataframe `df_data_1`: 

In [2]:
import pandas as pd

dataset_url = "https://drive.google.com/uc?export=download&id=1zo13kUAf-MDMPZmBDxq1FxWtZY01lsxD"
df_data_1 = pd.read_csv(dataset_url, sep='\t',  header=None, quoting=3)

Κάθε ομάδα θα δουλέψει σε **ένα μοναδικό υποσύνολο 5.000 ταινιών** (διαφορετικό dataset για κάθε ομάδα) ως εξής:

1. Κάθε ομάδα του εργαστηρίου νευρωνικών έχει έναν αριθμό στο helios. Θα βάλετε τον αριθμό αυτό στη μεταβλητή team_seed_number στο επόμενο κελί κώδικα.

2. Το data frame `df_data_2` έχει γραμμές όσες και οι ομάδες και 5.000 στήλες. Σε κάθε ομάδα αντιστοιχεί η γραμμή του πίνακα με το `team_seed_number` της. Η γραμμή αυτή θα περιλαμβάνει 5.000 διαφορετικούς αριθμούς που αντιστοιχούν σε ταινίες του αρχικού dataset. 

3. Τρέξτε τον κώδικα. Θα προκύψουν τα μοναδικά για κάθε ομάδα  titles, categories, catbins, summaries και corpus με τα οποία θα δουλέψετε.

In [3]:
import numpy as np

# Στο επόμενη γραμή βάλτε τον αριθμό της ομάδας στο εργαστήριο των νευρωνικών
team_seed_number = 4

movie_seeds_url = "https://drive.google.com/uc?export=download&id=1g6F4TCHrs2wgtdOk7D3gtONaeirNt_Vo"
df_data_2 = pd.read_csv(movie_seeds_url, header=None)

# επιλέγεται 
my_index = df_data_2.iloc[team_seed_number,:].values

titles = df_data_1.iloc[:, [2]].values[my_index] # movie titles (string)
categories = df_data_1.iloc[:, [3]].values[my_index] # movie categories (string)
bins = df_data_1.iloc[:, [4]]
catbins = bins[4].str.split(',', expand=True).values.astype(float)[my_index] # movie categories in binary form (1 feature per category)
summaries =  df_data_1.iloc[:, [5]].values[my_index] # movie summaries (string)
corpus = summaries[:,0].tolist() # list form of summaries
corpus_df = pd.DataFrame(corpus) # dataframe version of corpus

- Ο πίνακας **titles** περιέχει τους τίτλους των ταινιών. Παράδειγμα: 'Sid and Nancy'.
- O πίνακας **categories** περιέχει τις κατηγορίες (είδη) της ταινίας υπό τη μορφή string. Παράδειγμα: '"Tragedy",  "Indie",  "Punk rock",  "Addiction Drama",  "Cult",  "Musical",  "Drama",  "Biopic \[feature\]",  "Romantic drama",  "Romance Film",  "Biographical film"'. Παρατηρούμε ότι είναι μια comma separated λίστα strings, με κάθε string να είναι μια κατηγορία.
- Ο πίνακας **catbins** περιλαμβάνει πάλι τις κατηγορίες των ταινιών αλλά σε δυαδική μορφή ([one hot encoding](https://hackernoon.com/what-is-one-hot-encoding-why-and-when-do-you-have-to-use-it-e3c6186d008f)). Έχει διαστάσεις 5.000 x 322 (όσες οι διαφορετικές κατηγορίες). Αν η ταινία ανήκει στο συγκεκριμένο είδος η αντίστοιχη στήλη παίρνει την τιμή 1, αλλιώς παίρνει την τιμή 0.
- Ο πίνακας **summaries** και η λίστα **corpus** περιλαμβάνουν τις συνόψεις των ταινιών (η corpus είναι απλά ο summaries σε μορφή λίστας). Κάθε σύνοψη είναι ένα (συνήθως μεγάλο) string. Παράδειγμα: *'The film is based on the real story of a Soviet Internal Troops soldier who killed his entire unit  as a result of Dedovschina. The plot unfolds mostly on board of the prisoner transport rail car guarded by a unit of paramilitary conscripts.'*
- το dataframe **corpus_df** που είναι απλά το corpus σε μορφή dataframe. Τα summaries βρίσκονται στην κολόνα 0. Πιθανώς να σας βολεύει να κάνετε κάποιες προεπεξεργασίες με dataframes.


Θεωρούμε ως **ID** της κάθε ταινίας τον αριθμό γραμμής της ή το αντίστοιχο στοιχείο της λίστας. Παράδειγμα: για να τυπώσουμε τη σύνοψη της ταινίας με `ID=999` (την χιλιοστή) θα γράψουμε `print(corpus[999])`.

In [4]:
ID = 999
print(titles[ID])
print(categories[ID])
print(catbins[ID])
print(corpus[ID])

['Kabooliwala']
['"Romance Film",  "Drama",  "Comedy"']
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

# Εφαρμογή 1. Υλοποίηση συστήματος συστάσεων ταινιών βασισμένο στο περιεχόμενο
<img src="http://clture.org/wp-content/uploads/2015/12/Netflix-Streaming-End-of-Year-Posts.jpg" width="70%">

Η πρώτη εφαρμογή που θα αναπτύξετε θα είναι ένα [σύστημα συστάσεων](https://en.wikipedia.org/wiki/Recommender_system) ταινιών βασισμένο στο περιεχόμενο (content based recommender system). Τα συστήματα συστάσεων στοχεύουν στο να προτείνουν αυτόματα στο χρήστη αντικείμενα από μια συλλογή τα οποία ιδανικά θέλουμε να βρει ενδιαφέροντα ο χρήστης. Η κατηγοριοποίηση των συστημάτων συστάσεων βασίζεται στο πώς γίνεται η επιλογή (filtering) των συστηνόμενων αντικειμένων. Οι δύο κύριες κατηγορίες είναι η συνεργατική διήθηση (collaborative filtering) όπου το σύστημα προτείνει στο χρήστη αντικείμενα που έχουν αξιολογηθεί θετικά από χρήστες που έχουν παρόμοιο με αυτόν ιστορικό αξιολογήσεων και η διήθηση με βάση το περιεχόμενο (content based filtering), όπου προτείνονται στο χρήστη αντικείμενα με παρόμοιο περιεχόμενο (με βάση κάποια χαρακτηριστικά) με αυτά που έχει προηγουμένως αξιολογήσει θετικά.

Το σύστημα συστάσεων που θα αναπτύξετε θα βασίζεται στο **περιεχόμενο** και συγκεκριμένα στις συνόψεις των ταινιών (corpus). 


## Προεπεξεργασία

Το πρώτο βήμα στην επεξεργασία μας είναι ο καθαρισμός των περιγραφών των ταινιών. 

Εκτυπώστε (αρκετές) διαφορετικές περιγραφές ταινιών για να δείτε πιθανά προβλήματα που θα πρέπει να αντιμετωπιστούν.

Τα (ελάχιστα) βήματα καθαρισμού που προτείνουμε είναι:
- μετατροπή όλων των χαρακτήρων σε πεζά,
- αφαίρεση των stopwords. Εδώ σημειώστε ότι για το δεδομένο task του συστήματος συστάσεων που είναι η πρόταση ταινιών ίσως θα είχαν ενδιαφέρον και λίστες stopwords πέραν αυτών της κοινής γλώσσας.
- αφαίρεση σημείων στίξης και ειδικών χαρακτρήρων (special characters). Αυτό δεν γίνεται μόνο με την punkt του NLTK. Θα μπορούσατε να βασιστείτε σε κανονικές εκφράσεις (regular expressions), και
- αφαίρεση πολυ σύντομων συμβολοσειρών.

Προσοχή: το corpus και τα τελικά tokens που θα το αποτελούν θα χρησιμοποιηθούν στη συνέχεια ως κλειδιά για να βρούμε εμφυτεύματα. Για το λόγο αυτό, πρέπει να είστε προσεκτικοί ως προς την εφαρμογή μεθόδων κανονικοποίησης (text normalization) όπως το stemming και το lemmatization.

In [5]:
# Αρχικά τυπώνουμε 10 διαφορετικές περιγραφές ταινιών για να εξετάσουμε τα corpus τους

for i in range(0,10):
  print(corpus[i])
  print("")

# Μπορούμε να αλλάξουμε τα όρια στο range ανά πάσα στιγμή, έτσι ώστε να εξετάσουμε ένα διαφορετικό σύνολο περιγραφών

Ambrose Wolfinger works as a "memory expert" for a manufacturing company's president; he keeps files of details about all the people President Malloy meets with, so that Malloy will never be embarrassed about not remembering things when meeting with them. Ambrose supports himself, his shrewish wife Leona, his loving daughter Hope , his freeloading brother-in-law Claude, and his abusive mother-in-law Cordelia. At the start of the film, two burglars break into Ambrose's cellar late at night, get drunk on his homemade applejack, and start singing "On the Banks of the Wabash"; Ambrose is forced to handle the situation, and he winds up being arrested for distilling liquor without a license. The next day, Ambrose  tells Malloy that Cordelia had died from drinking poisoned liquor, and asks for the afternoon off to attend the funeral; in fact, he wants to go to see the big wrestling match. Malloy, touched by Ambrose's tale, lets him go for the day, and Ambrose's immediate supervisor, Mr. Peabo

In [6]:
!pip install --upgrade contractions

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.1/110.1 kB 11.5 MB/s eta 0:00:00


In [7]:
# Ξεκινάμε την προεπεξεργασία
import nltk
import urllib
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import string
nltk.download('omw-1.4')
nltk.download('wordnet') 
nltk.download('rslp')
from nltk.stem import WordNetLemmatizer
import re
import contractions

# Λέξεις που εμφανίζονται συχνά στις περιγραφές ταινιών και επομένως δυσχεραίνουν την ορθή λειτουργία του recommender
common_descWords = ("movie", "plot", "protagonist", "film", "story", "finale", "summary", "'s")

wordnet_lemmatizer = WordNetLemmatizer()
punctuations = list(string.punctuation)
st = set(stopwords.words("english"))

def thorough_filter(words):
    filtered_words = []
    for word in words:
        pun = []
        for letter in word:
            pun.append(letter in punctuations)
        if not all(pun):
            filtered_words.append(word)
    return filtered_words

def get_names(url):
    decoded_url = urllib.request.urlopen(url).read().decode()
    name_list = decoded_url.split('\n')
    
    names = []
    for name in name_list:
        if name and name[0] != '#': # αγνοούμε τις πρώτες γραμμές που ξεκινάνε με 'δίεαση'
            names.append(name.lower())
    return set(names)

male_names = get_names("http://www.cs.cmu.edu/Groups/AI/util/areas/nlp/corpora/names/male.txt")
female_names = get_names("http://www.cs.cmu.edu/Groups/AI/util/areas/nlp/corpora/names/female.txt")

st = st.union(st, common_descWords)
st = st.union(st, male_names)
st = st.union(st, female_names)

for i in range(0,len(corpus)):
  corpus[i] = corpus[i].lower()  #μετατροπή των χαρακτήρων σε πεζά
  corpus[i] = re.sub(r"http\S+", " ", corpus[i])   #αφαιρούμε τα url links
  corpus[i] = contractions.fix(corpus[i])  #αφαιρούμε τις συντομογραφίες don't -> do not, you're -> you are
  tok = word_tokenize(corpus[i])
  tok = [i for i in tok if i not in punctuations] #αφαίρεση των ειδικών χαρακτήρων και της στίξης
  tok = thorough_filter(tok) #δεύτερο φιλτράρισμα για την περίπτωση που μια λέξη περιέχει punctuation
  tok = [word for word in tok if len(word)>2] #αφαίρεση των λέξεων με λιγότερους από 2 χαρακτήρες
  tok = [word for word in tok if not word in st] #αφαίρεση των stopwords, των ονομάτων, και των κοινών λέξεων
  corpus[i] = " ".join(wordnet_lemmatizer.lemmatize(word) for word in tok) #εφαρμόζουμε lemmatization

# Ελέγχουμε ότι δούλεψε
for i in range(0,10):
  print(corpus[i])
  print("")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


wolfinger work memory expert manufacturing company president keep file detail people president malloy meet malloy never embarrassed remembering thing meeting support shrewish wife loving daughter freeloading brother-in-law abusive mother-in-law start two burglar break cellar late night get drunk homemade applejack start singing bank wabash forced handle situation wind arrested distilling liquor without license next day tell malloy died drinking poisoned liquor asks afternoon attend funeral fact want big wrestling match malloy touched tale let day immediate supervisor mr. peabody tell employee tragic news pay respect family throughout day one problem another encounter ticket-writing policeman car parked close find chasing tire along railroad track narrowly avoids getting hit train trying get wrestling arena get knocked wrestler thrown building opponent later day come home find furious seeing obituary newspaper receiving huge amount flower sympathy card funeral wreath furthermore peabody

Στην προεπεξεργασία εκτελέσαμε όλα τα βήματα που προτείνονται στην εκφώνηση (μετατροπή όλων των χαρακτήρων σε πεζά γιατί ο αλγόριθμος που συγκρίνει τις λέξεις μεταξύ τους είναι case sensitive, και αφαίρεση stopwords, πολύ μικρών συμβολοσειρών, σημείων στίξεως, και ειδικών χαρακτήρων καθώς δεν περιέχουν κάποια ουσιαστική πληροφρορία για την πλοκή μιας ταινίας αλλά η συχνότητα εμφάνισής τους μπορεί να επηρεάσει την μετρική tf-idf). Για τον δεύτερο από τους δύο λόγους που μόλις αναφέραμε, επιλέξαμε επίσης να αφαιρέσουμε όσα πιο πολλά ονόματα χαρακτήρων μπορούσαμε, καθώς και κάποιες λέξεις όπως "movie", "plot", κτλ οι οποίες εμφανίζονται συχνά σε περιγραφές ταινιών αλλά δεν έχουν άμεση σχέση με την πλοκή. Υπήρξε σκέψη και για αφαίρεση των αριθμών, αλλά αποφασίσαμε να μην το κάνουμε, καθώς κάποιοι αριθμοί θα ήταν και ημερομηνίες σχετικές με την υπόθεση (πχ δυό ταινίες που διαδραματίζονται την ίδια χρονική περίοδο). 

Επιπλέον, επιλέγουμε να εφαρμόσουμε την μέθοδο lemmatization αντί της stemming, καθώς αυτή ενδέχεται να εμφανίσει προβλήματα στην αναπαράσταση με word embeddings.

## Μετατροπή σε TFIDF

Το πρώτο βήμα θα είναι λοιπόν να μετατρέψετε το corpus σε αναπαράσταση tf-idf:

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
# create sparse tf_idf representation
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)
corpus_tf_idf_plain = vectorizer.transform(corpus).toarray()

Η συνάρτηση [TfidfVectorizer](http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) όπως καλείται εδώ **δεν είναι βελτιστοποιημένη**. Οι επιλογές των μεθόδων και παραμέτρων της μπορεί να έχουν **δραματική επίδραση στην ποιότητα των συστάσεων** και είναι διαφορετικές για κάθε dataset. Επίσης, οι επιλογές αυτές έχουν πολύ μεγάλη επίδραση και στη **διαστατικότητα και όγκο των δεδομένων**. Η διαστατικότητα των δεδομένων με τη σειρά της θα έχει πολύ μεγάλη επίδραση στους **χρόνους εκπαίδευσης**, ιδιαίτερα στη δεύτερη εφαρμογή της άσκησης.

Προσοχή: ο TfidfVectorizer έχει κάποιες δυνατότητες προεπεξεργασίας παρόποιες με αυτές που αναφέραμε στην προηγούμενη ενότητα. Ό,τι προεπεξεργασία μπορείτε να κάνετε που χρειάζεται ως είσοδο μόνο το κάθε document ξεχωριστά, κάντε την στο πρώτο βήμα της προεπεξεργασίας. Αν χρειάζεται γνώση των συνολικών στατιστικών της συλλογής, κάντε την με τον TfidfVectorizer.

In [9]:
print(corpus_tf_idf_plain.shape)

(5000, 41528)


## Υλοποίηση του συστήματος συστάσεων

Το σύστημα συστάσεων που θα υλοποιήσετε θα είναι μια συνάρτηση `content_recommender` με τρία ορίσματα: `target_movie`, `max_recommendations` και `corpus_type`. Στην `target_movie` περνάμε το ID μιας ταινίας-στόχου για την οποία μας ενδιαφέρει να βρούμε παρόμοιες ως προς το περιεχόμενο (τη σύνοψη) ταινίες, `max_recommendations` στο πλήθος.
Υλοποιήστε τη συνάρτηση ως εξής: 
- για την ταινία-στόχο, θα υπολογίζετε την [ομοιότητα συνημιτόνου](https://en.wikipedia.org/wiki/Cosine_similarity) της με όλες τις ταινίες της συλλογής σας όπως αυτές αναπαριστώνται στο `corpus_type`.
- με βάση την ομοιότητα συνημιτόνου που υπολογίσατε, δημιουργήστε ταξινομημένο πίνακα από το μεγαλύτερο στο μικρότερο, με τα indices (`ID`) των ταινιών. Παράδειγμα: αν η ταινία με index 1 έχει ομοιότητα συνημιτόνου με 3 ταινίες \[0.2 1 0.6\] (έχει ομοιότητα 1 με τον εαύτό της) ο ταξινομημένος αυτός πίνακας indices θα είναι \[1 2 0\].
- Για την ταινία-στόχο εκτυπώστε: id, τίτλο, σύνοψη, κατηγορίες (categories)
- Για τις `max_recommendations` ταινίες (πλην της ίδιας της ταινίας-στόχου που έχει cosine similarity 1 με τον εαυτό της) με τη μεγαλύτερη ομοιότητα συνημιτόνου (σε φθίνουσα σειρά), τυπώστε σειρά σύστασης (1 πιο κοντινή, 2 η δεύτερη πιο κοντινή κλπ), ομοιότητα συνημιτόνου, id, τίτλο, σύνοψη, και κατηγορίες (categories)


In [10]:
import numpy as np
from numpy import linalg as LA
import scipy as sp

def content_recommender(target_movie, max_recommendations, corpus_type):
  #tar_list = [corpus[target_movie]]
  #vectorizer.fit(tar_list)
  #target_tfidf = vectorizer.transform(tar_list)
  cossim = []
  for i in range(len(corpus_type)):
    #np.reshape(corp_tfidf[target_movie], corp_tfidf[i].shape)
    #cossim[i] = np.dot(target_tfidf[0],corp_tfidf[i])/(LA.norm(target_tfidf[0])*LA.norm(corp_tfidf[i])) #υπολογισμός ομοιότητας συνημιτόνου
    cossim.append(sp.spatial.distance.cosine(corpus_type[target_movie], corpus_type[i]))
  sorted = np.argsort(cossim)[::-1]
  print("Target movie ID:",target_movie)
  print("Target movie title:",titles[target_movie])
  print("Target movie description:",corpus[target_movie])
  print("Target movie categories:",categories[target_movie])
  for i in range(1, max_recommendations + 1):
    mov_id = sorted[i]
    print("Σειρά σύστασης:", i)
    print("Ομοιότητα συνημιτόνου:",cossim[mov_id])
    print("Recommendation movie ID:",mov_id)
    print("Recommendation movie title:",titles[mov_id])
    print("Recommendation movie description:",corpus[mov_id])
    print("Recommendation movie categories:",categories[mov_id])

## Βελτιστοποίηση του TfidfVectorizer

Αφού υλοποιήσετε τη συνάρτηση `content_recommender` χρησιμοποιήστε την για να βελτιστοποιήσετε την `TfidfVectorizer`. Συγκεκριμένα, αρχικά μπορείτε να δείτε τι επιστρέφει το σύστημα για τυχαίες ταινίες-στόχους και για ένα μικρό `max_recommendations` (2 ή 3). Αν σε κάποιες ταινίες το σύστημα μοιάζει να επιστρέφει σημασιολογικά κοντινές ταινίες σημειώστε το `ID` τους. Δοκιμάστε στη συνέχεια να βελτιστοποιήσετε την `TfidfVectorizer` για τα συγκεκριμένα `ID` ώστε να επιστρέφονται σημασιολογικά κοντινές ταινίες για μεγαλύτερο αριθμό `max_recommendations`. Παράλληλα, όσο βελτιστοποιείτε την `TfidfVectorizer`, θα πρέπει να λαμβάνετε καλές συστάσεις για μεγαλύτερο αριθμό τυχαίων ταινιών. 

Ταυτόχρονα, μια αντίρροπη κατά κάποιο τρόπο κατεύθυνση της βελτιστοποίησης είναι να χρησιμοποιείτε τις παραμέτρους του `TfidfVectorizer` έτσι ώστε να μειώνονται οι διαστάσεις του Vector Space Model μέχρι το σημείο που θα αρχίσει να εμφανίζονται επιπτώσεις στην ποιότητα των συστάσεων. 




### **Test 1**

In [11]:
content_recommender(1,3,corpus_tf_idf_plain)

Target movie ID: 1
Target movie title: ['A Prize of Arms']
Target movie description: three criminal hatched plan army barrack troop dispatched take part war middle east believed large amount pay premise shipped gang enters army barrack disguised soldier proceeds pay corp headquarters guise maintenance work make sure alarm disabled give time make escape robbery take place rest day try integrate working base including vaccinated overseas service avoid attracting attention night fall change military police uniform head pay headquarters announcing arrival report fire begin searching room starting small blaze order premise evacuated building empty break safe steal £100,000 starting several fire cover activity withdraw carrying fake casualty stretcher troop rush across base put fire men drive secluded spot base left army truck officer ring medic check progress casualty told nobody arrived suspicious raise alarm whole camp put standby police sent initially fooled thinking criminal already lef

Για αυτήν την περίπτωση, η 3η σε σειρά σύσταση είναι πολύ καλή γιατί είναι και αυτή ένα thriller που αφορά εγκληματίες. Η 2η σύσταση έχει κάποια συνάφεια, καθώς είναι και αυτή thriller. Η 1η όμως σύσταση δεν είναι σχετική, καθώς είναι ταινία επιστημονικής φαντασίας σχετική με εξωγήινους.

### **Test 2**

In [12]:
content_recommender(10,3,corpus_tf_idf_plain)

Target movie ID: 10
Target movie title: ['Rabbit Transit']
Target movie description: relaxing steam bath bug read original fable reading credit tortoise beat hare becomes incensed idea turtle outrunning rabbit also steam bath claim could outrun bug prompting bug challenge race time bug agree cheating however quickly reveals rocket propelled allowing surprising combination fast slow bug best steal dismantle destroy device little effect end however bug manage top turtle cross finish line first nevertheless last laugh rook rabbit confessing 100 easy —in speed zone bug taken away police enjoy victory—behind bar close cartoon famously saying stinker
Target movie categories: ['"Short Film",  "Comedy film",  "Family Film",  "Animation"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 1869
Recommendation movie title: ['When Strangers Marry']
Recommendation movie description: naive woman come new york city meet salesman husband met month discovers murderer
Recommendation 

Σε αυτήν την περίπτωση και οι 3 συστάσεις είναι κακές, αφού το target movie είναι μια ταινία κινουμένων σχεδίων για παιδιά, ενώ οι συστάσεις είναι δύο thriller και μία τανία δράσης-φαντασίας που δεν είναι κατάλληλη για παιδιά. 

### **Test 3**

In [13]:
content_recommender(30,6,corpus_tf_idf_plain)

Target movie ID: 30
Target movie title: ['Cover Story']
Target movie description: khan tabu next door neighbour chandrashekar retired judge haunted past two become friend computer engineer wear high power contact lens vision problem meet news reporter executive director true vision chandrashekar killed mysterious man seen recognised wearing contact lens time corrupt policeman think killer eventually discovers actually killer also killed chandashekar search real killer police attempting catch form rest
Target movie categories: ['"Thriller",  "Crime Fiction",  "World cinema",  "Drama",  "Romantic drama",  "Crime Thriller",  "Romance Film"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 1265
Recommendation movie title: ['Fleshtone']
Recommendation movie description: painter play erotic game telephone woman body found mutilated
Recommendation movie categories: ['"Thriller"']
Σειρά σύστασης: 2
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 4597
Recommendation mo

Εδώ οι συστάσεις είναι ικανοποιητικές, καθώς σχεδόν όλες ανήκουν σε κάποιο genre στο οποίο ανήκει και το target movie. Συγκεκριμένα η 1η είναι και αυτή thriller σχετικό με εγκλήματα, η 2η περιλαμβάνει έναν φόνο, η 3η είναι και αυτή δραματική και ρομαντική, η 4η περιέχει και αυτή βία, η 5η είναι και αυτή δραματική και σχετική με εγγλήματα, και η 6η είναι και αυτή ρομαντικό δράμα και σχετική με τον υπόκοσμο. 

### **Test 4**

In [39]:
content_recommender(300,5,corpus_tf_idf_plain)

Target movie ID: 300
Target movie title: ['A Bittersweet Life']
Target movie description: sun-woo high ranking mobster enforcer kang cold calculating crime bos unquestionably loyal two share concern business tension baek jr. rival family kang assigns sun-woo perceived simple errand away business young mistress heesoo fear affair another man giving sun-woo mandate kill manages discover performs duty following heesoo escorting music recital one day becomes quietly enthralled girl beauty innocence glimpse lonely empty personal life become prevalent come discover heesoo lover directly home fiercely beat prepares inform kang attraction cause hesitate thus spare two condition longer earning heesoo enmity meanwhile sun-woo continues embroiled personal business baek jr. beaten several henchman earlier overstaying welcome hotel threatened one enforcer apologize adamantly refuse fueled frustration heesoo relaxes apartment later one night suddenly kidnapped baek men tortured receive new order via

Σε αυτήν την περίπτωση όλες οι συστάσεις είναι κακές καθώς είναι εντελώς άσχετες με την ταινία στόχο. 

### **Test 5**

In [43]:
content_recommender(444,5,corpus_tf_idf_plain)

Target movie ID: 444
Target movie title: ['Is There Life Out There']
Target movie description: loving supportive husband two great kid unfulfilled dream return college get degree always wanted life beyond family home wonder hole life soon filled much confusing new social life campus schoolwork keeping late part-time keeping husband kid whose mom turning stranger strength perseverance thing help
Target movie categories: ['"Drama",  "Television movie"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 4398
Recommendation movie title: ['Screaming Mimi']
Recommendation movie description: opening scene set southern california taking outside beach shower escaped madman sanitarium show stab dog devil name second dog attack shot death stepbrother rifle attack committed sanitarium psychiatrist fall fake death lam end dancing madhouse night club run performs put blame classic noir theme stalked serial killer
Recommendation movie categories: ['"Thriller",  "Psychological thri

(Σχόλια)

Συνολικά, μπορούμε να δούμε πως ο recommender μας δουλεύει αρκετά καλά για τις μισές περίπου περιπτώσεις. Έτσι θα προβούμε στην βελτιστοποίση του. 

Δοκιμάζουμε να χρησιμοποιήσουμε την παράμετρο strip_accents, έτσι ώστε δυο λέξεις που είναι ίδιες σε δυο γλώσσες με την εξέρεση ενός τόνου να αναγνωριστούν ως ίδιες σημασιολογικά (φυσικά αυτό επιφέρει τον κίνδυνο να είναι διαφορετικές σημασιολογικά στην πραγματικότητα, αλλά συνήθως συμβαίνει το αντίθετο). Ταυτόχρονα δεν λαμβάνουμε υπόψην τις λέξεις που εμφανίζονται σε 90% των περιγραφών ή παραπάνω ή μόνο μία φορά, καθώς προφανώς είναι λέξεις που απλά μας "ξέφυγαν" από την προεπεξεργασία και δεν έχουν σημασία για το recommendation. 

In [14]:
vectorizer_2 = TfidfVectorizer(strip_accents= 'unicode', max_df=0.9, min_df=1)
vectorizer_2.fit(corpus)
corpus_tf_idf_2 = vectorizer_2.transform(corpus).toarray()
print(corpus_tf_idf_plain.shape)
print(corpus_tf_idf_2.shape)

(5000, 41528)
(5000, 41418)


Επαναλαμβάνουμε τώρα τα προηγούμενα Test.

### **Test 1**

In [15]:
content_recommender(1,3,corpus_tf_idf_2)

Target movie ID: 1
Target movie title: ['A Prize of Arms']
Target movie description: three criminal hatched plan army barrack troop dispatched take part war middle east believed large amount pay premise shipped gang enters army barrack disguised soldier proceeds pay corp headquarters guise maintenance work make sure alarm disabled give time make escape robbery take place rest day try integrate working base including vaccinated overseas service avoid attracting attention night fall change military police uniform head pay headquarters announcing arrival report fire begin searching room starting small blaze order premise evacuated building empty break safe steal £100,000 starting several fire cover activity withdraw carrying fake casualty stretcher troop rush across base put fire men drive secluded spot base left army truck officer ring medic check progress casualty told nobody arrived suspicious raise alarm whole camp put standby police sent initially fooled thinking criminal already lef

Εδώ η δεύτερη σύσταση είναι καλή αφού είναι βίαιη ταινία που αφορά εγκληματίες. Οι άλλες δύο όμως είναι άσχετες.

### **Test 2**

In [16]:
content_recommender(10,3,corpus_tf_idf_2)

Target movie ID: 10
Target movie title: ['Rabbit Transit']
Target movie description: relaxing steam bath bug read original fable reading credit tortoise beat hare becomes incensed idea turtle outrunning rabbit also steam bath claim could outrun bug prompting bug challenge race time bug agree cheating however quickly reveals rocket propelled allowing surprising combination fast slow bug best steal dismantle destroy device little effect end however bug manage top turtle cross finish line first nevertheless last laugh rook rabbit confessing 100 easy —in speed zone bug taken away police enjoy victory—behind bar close cartoon famously saying stinker
Target movie categories: ['"Short Film",  "Comedy film",  "Family Film",  "Animation"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 264
Recommendation movie title: ['Fast Girls']
Recommendation movie description: follows athlete shania andrew competes local level follow duo work british 4x200 metre relay team compete wo

Εδώ η πρώτη σύσταση βελτιώθηκε, καθώς είναι μία ταινία την οποία θα μπορούσε να παρακολουθήσει ένα παιδί και έχει σχέση με το τρέξιμο όπως και η ταινία στόχος. Οι δύο επόμενες συστάσεις εξακολουθούν να είναι κακές. 

### **Test 3**

In [17]:
content_recommender(30,6,corpus_tf_idf_2)

Target movie ID: 30
Target movie title: ['Cover Story']
Target movie description: khan tabu next door neighbour chandrashekar retired judge haunted past two become friend computer engineer wear high power contact lens vision problem meet news reporter executive director true vision chandrashekar killed mysterious man seen recognised wearing contact lens time corrupt policeman think killer eventually discovers actually killer also killed chandashekar search real killer police attempting catch form rest
Target movie categories: ['"Thriller",  "Crime Fiction",  "World cinema",  "Drama",  "Romantic drama",  "Crime Thriller",  "Romance Film"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0
Recommendation movie ID: 3762
Recommendation movie title: ['Ajantha']
Recommendation movie description: vinu ramana play upcoming musician finishing recording home see artist creating street stricken awe onlooker giving artist rupee much artist expected point begin rain drawing begin dissolve onlooker dispe

Εδώ η 1η και η 3η συστάσεις είναι διαφορετικές σε σχέση με την προηγούμενη φορά, παρέμειναν όμως σχετικές (η πρώτη είναι ρομαντικό δράμα και η 3η δράμα). 


### **Test 4**

In [40]:
content_recommender(300,5,corpus_tf_idf_2)

Target movie ID: 300
Target movie title: ['A Bittersweet Life']
Target movie description: sun-woo high ranking mobster enforcer kang cold calculating crime bos unquestionably loyal two share concern business tension baek jr. rival family kang assigns sun-woo perceived simple errand away business young mistress heesoo fear affair another man giving sun-woo mandate kill manages discover performs duty following heesoo escorting music recital one day becomes quietly enthralled girl beauty innocence glimpse lonely empty personal life become prevalent come discover heesoo lover directly home fiercely beat prepares inform kang attraction cause hesitate thus spare two condition longer earning heesoo enmity meanwhile sun-woo continues embroiled personal business baek jr. beaten several henchman earlier overstaying welcome hotel threatened one enforcer apologize adamantly refuse fueled frustration heesoo relaxes apartment later one night suddenly kidnapped baek men tortured receive new order via

Πλέον, γι αυτήν την ταινία υπάρχουν δύο σχετικές συστάσεις (η 3η και η 5η), επομένως υπάρχει εμφανής βελτίωση. 

Παρατηρούμε πως η λειτουργία του recommender βελτιώνεται, δεν αλλάζει δραματικά. Όμως δεν υπάρχουν και άλλες βελτιώσεις που θα μπορούσαμε να κάνουμε, καθώς τις κάναμε ήδη στην προεπεξεργασία (πχ. αφαίρεση των stopwords, μετατροπή όλων των χαρακτήρων σε πεζά). Πράγματι όμως τώρα  προτείνονται περισσότερες τανίες με μη αγγλικές λέξεις στις περιγραφές τους, οι οποίες όμως εξακολουθούν να είναι σχετικές με την τανία-στόχο.  


### **Γενικές παρατηρήσεις για την λειτουργία του recommender**


**Cherry-Picking:** Κάποιες ταινίες στόχοι για τις οποίες επιστρέφονται καλά αποτελέσματα είναι οι 

**Nit-Picking:** Κάποιες ταινίες στόχοι για τις οποίες επιστρέφονται κακά αποτελέσματα είναι αυτές με ID 30 και 300. Γι αυτές τις ταινίες δεν επιτύχαμε καμία καλή σύσταση πριν από την βελτιστοποίηση, αλλά ακόμη και μετά από αυτήν οι καλές συστάσεις ήταν μόνο λίγες. 

**Συνολικά πλεονεκτήματα και μειονεκτήματα ενός recommender βασισμένου στο tfidf:** Ένας recommender που είναι βασισμένος στο tfidf είναι αρκετά απλός στην υλοποιήση. Όμως, δεν δίνει πολύ καλά αποτελέσματα χωρίς βελτιστοποίηση, και ακόμη και τότε είναι αρκετά πιο πιθανό κάποιες από τις συστάσεις να είναι κακές από το να είναι όλες καλές. 


----

Μία ακόμη γενική παρατήρηση που πρέπει να γίνει είναι πως η καλύτερη σύσταση δεν είναι αναγκαστικά η πρώτη, και αυτό είναι λογικό καθώς οι πρώτες x συστάσεις για κάθε ταινία (με x>max_recommendation για όσες έχουμε τυπώσει) έχουν όλες το ίδιο cosine similarity (ίσο με 1.0), επομένως δεν μπορούν να ταξινομθούν μεταξύ τους με αποτελεσματικό τρόπο. 

## Βαθιά μάθηση: δημιουργία corpora με χρήση word emmbeddings

Η προσέγγιση της κατασκευής μόνο μέσω tfidf του συστήματος συστάσεων έχει διάφορα μειονεκτήματα. Θα μας ενδιέφερε λοιπόν να δούμε αν μπορούμε να χρησιμοποιήσουμε για τις λέξεις **εμφυτεύματα (embeddings)**, δηλαδή τις πυκνές διανυσματικές αναπαραστάσεις για τις λέξεις που μας δίνει το μοντέλο **Word2Vec**

Ωστόσο, το dataset της κάθε ομάδας είναι πολύ μικρό για να εξάγουμε τα δικά μας word embeddings (και να είναι καλά). Για το λόγο αυτό θα χρησιμοποιήσουμε τη μεθοδολογία της Βαθιάς Μάθησης που είναι η **Μεταφορά Μάθησης (Transfer Learning).**.

Στη μεταφορά μάθησης ουσιαστικά μεταφέρουμε τη γνώση που έχει αποκτήσει ένα ήδη εκπαιδευμένο (και κατά κανόνα πολύ μεγάλο) σύστημα. Η μεταφορά γίνεται διαμέσου των τιμών των βαρών που έχει προσδιορίσει μετά το πέρας της εκπαίδευσης.

Στην περίπτωσή μας, δεν μας ενδιαφέρουν τόσο τα ίδια τα βάρη των μοντέλων από τα οποία θα κάνουμε μεταφορά μάθησης. Κάτι τέτοιο θα μας ενδιέφερε αν π.χ. θέλαμε να συνεχίσουμε την εκπαίδευση στα δικά μας κείμενα. Μας ενδιαφέρουν όμως τα ίδια τα εμφυτεύματα, δηλαδή τα embeddings (διανύσματα διαστάσεων $m$) που έχει μάθει το νευρωνικό για το λεξιλόγιο του (vocabulary). To vocabulary σε τέτοια μεγάλα νευρωνικά θα είναι πιθανότατα υπερσύνολο του δικού μας.

### Μεταφορά μάθησης εμφυτευμάτων



#### Εμφυτεύματα του Gensim-data
Το Gensim περιλαμβάνει αρκετά προεκπαιδευμένα μοντέλα εμφυτευμάτων Word2Vec. Με το επόμενο κελί παίρνουμε τη λίστα τους.

In [18]:
!pip install -U gensim
import gensim.downloader as api
print(list(api.info()['models'].keys()))

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 48.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for fst-pso: filename=fst_pso-1.8.1-py3-none-any.whl size=20443 sha256=eb1e8c18ba4b094dd5666ecf303f27686760da9e57377779cf416fa17fbe66a3
  Stored in directory: /root/.cache/pip/wheels/b4/d2/63/953ff58cf13a31f09d49568238da542cfaedbc8d9cd0adbf80
  Created wheel for miniful: filename=miniful-0.0.6-py3-none-any.whl size=3530 sha256=3cb83fec8a3cac196ec6659d460d22bc5a8e595c847d780a00a82af716a35d91
  Stored in directory: /root/.cache/pip/wheels/26/26/35/93cbca6b31bba14406cde5b9c6f6f61b93dfe0ce68ca084d25
Successfully built fst-pso miniful
  Attempting uninstall: gensim
    Found existing installation: gensim 3.6.0
    Uninstalling gensim-3.6.0

Τα μοντέλα αυτά βρίσκονται στο [αποθετήριο Gensim-data](https://github.com/RaRe-Technologies/gensim-data) όπου μπορείτε να βρείτε και την τεκμηρίωσή τους. Η φόρτωση των μοντέλων αυτών γίνεται με τη συνάρτηση `gensim.downloader.load`.

Επιλέγουμε το προεκπαιδευμένο μοντέλο glove-twitter-100, όπου για κάθε λέξη περιέχει τις 100 πιο παρομόμοιες λέξεις βασισμένο στο twitter.

In [19]:
model1 = api.load("glove-twitter-100") 

[==================================================] 100.0% 387.1/387.1MB downloaded


Βλέπουμε παρακάτω τις πιο όμοιες λέξεις ως προς την λέξη γάτα, έτσι ώστε να πιστοπιήσουμε πως δούλεψε.

In [20]:
model1.most_similar('cat')

[('dog', 0.8752089142799377),
 ('kitty', 0.8015091419219971),
 ('pet', 0.7986468076705933),
 ('cats', 0.797942578792572),
 ('kitten', 0.7936834096908569),
 ('puppy', 0.7702749967575073),
 ('monkey', 0.7584263682365417),
 ('bear', 0.7507943511009216),
 ('dogs', 0.746006190776825),
 ('pig', 0.7117345929145813)]

Κατεβάζουμε και κάποια ακόμη προεκπαιδευμένα μοντέλα. 

In [21]:
model2 = api.load('glove-wiki-gigaword-100')
model3 = api.load('glove-twitter-25')

[==================================================] 100.0% 128.1/128.1MB downloaded
[==================================================] 100.0% 104.8/104.8MB downloaded


In [22]:
print(model2.most_similar('cat'))
model3.most_similar('cat')

[('dog', 0.8798074722290039), ('rabbit', 0.7424427270889282), ('cats', 0.732300341129303), ('monkey', 0.7288709878921509), ('pet', 0.719014048576355), ('dogs', 0.7163872718811035), ('mouse', 0.6915250420570374), ('puppy', 0.6800068020820618), ('rat', 0.6641027331352234), ('spider', 0.6501135230064392)]


[('dog', 0.9590820074081421),
 ('monkey', 0.920357882976532),
 ('bear', 0.9143136739730835),
 ('pet', 0.9108031392097473),
 ('girl', 0.8880629539489746),
 ('horse', 0.8872726559638977),
 ('kitty', 0.8870542049407959),
 ('puppy', 0.886769711971283),
 ('hot', 0.886525571346283),
 ('lady', 0.8845519423484802)]

In [31]:
print(model2[1])

#Embedding με διάσταση 100

[-0.10767    0.11053    0.59812   -0.54361    0.67396    0.10663
  0.038867   0.35481    0.06351   -0.094189   0.15786   -0.81665
  0.14172    0.21939    0.58505   -0.52158    0.22783   -0.16642
 -0.68228    0.3587     0.42568    0.19021    0.91963    0.57555
  0.46185    0.42363   -0.095399  -0.42749   -0.16567   -0.056842
 -0.29595    0.26037   -0.26606   -0.070404  -0.27662    0.15821
  0.69825    0.43081    0.27952   -0.45437   -0.33801   -0.58184
  0.22364   -0.5778    -0.26862   -0.20425    0.56394   -0.58524
 -0.14365   -0.64218    0.0054697 -0.35248    0.16162    1.1796
 -0.47674   -2.7553    -0.1321    -0.047729   1.0655     1.1034
 -0.2208     0.18669    0.13177    0.15117    0.7131    -0.35215
  0.91348    0.61783    0.70992    0.23955   -0.14571   -0.37859
 -0.045959  -0.47368    0.2385     0.20536   -0.18996    0.32507
 -1.1112    -0.36341    0.98679   -0.084776  -0.54008    0.11726
 -1.0194    -0.24424    0.12771    0.013884   0.080374  -0.35414
  0.34951   -0.7226     0.

#### Άλλα εμφυτεύμαατα
Μπορείτε να βρείτε προεκπαιδευμένα εμφυτεύματα και από πηγές εκτός του Gensim. Για παράδειγμα:

- [Google News dataset](https://code.google.com/archive/p/word2vec/). Πρόκειται για προ-εκπαιδευμένα διανύσματα που έχουν εκπαιδευτεί σε μέρος του συνόλου δεδομένων Google News (περίπου 100 δισεκατομμύρια λέξεις). Το μοντέλο περιέχει διανύσματα 300 διαστάσεων για 3 εκατομμύρια λέξεις και φράσεις.
- [Amazon BlazingText](https://docs.aws.amazon.com/sagemaker/latest/dg/blazingtext.html). Το BlazingText δεν είναι μόνο προεκπαιδευμένα εμφυτεύματα αλλα και βελτιστοποιημένες υλοποιήσεις των αλγορίθμων Word2vec για την επεξεργασία κειμένου. Προυπόθεση είναι να δουλέψει κανείς στο SageMaker.

Οι διαδικασίες φόρτωσης embeddings από εξωτερικά δεδομένα μπορεί να είναι ελαφρά διαφορετικές από αυτή του Gensim.



#### Παρατηρήσεις

*   Επαναλαμβάνουμε ότι στην εργασία αυτή δεν μας ενδιαφέρουν τα ίδια τα μοντέλα αλλά το να μπορούμε για μία λέξη του λεξιλογίου μας να μπορούμε να βρούμε το embedding (διάνυσμα) που της αντιστοιχεί στο εκάστοτε προεκπαιδευμένο μοντέλο. 

*   Επίσης, δεν θα χρησιμοποιήσουμε την `Phrases` για να βρούμε bigrams στο dataset μας όπως θα ήταν το ορθότερο, καθώς αυτό θα απαιτούσε την συνέχιση της εκπαίδευσης του μοντέλου σε νέο λεξιλόγιο με πολύ λίγα νέα δεδομένα.


 ### Δημιουργία corpora βασισμένων στα εμφυτεύματα

Για να μπορέσουμε να ενσωματώσουμε τη γνώση που υπάρχει στα προεκπαιδευμένα εμφυτεύματα στο δικό μας corpus θα προχωρήσουμε όπως περιγράφεται ακολούθως.

Για κάθε περιγραφή ταινίας $d$, η οποία αποτελείται από τις $N_d$ λέξεις $w_i$, το  $tfidf$ της κάθε λέξης $w_i$ δίνεται από τη σχέση:

$$ tfidf(w_i) = tf(w_i,d) \cdot idf(w_i)$$

Ταυτόχρονα, σε κάθε λέξη $w_i$ αντιστοιχεί ένα διάνυσμα $W2V(w_i)$ από το μοντέλο εμφυτευμάτων που έχουμε εισάγει. Τα διανύσματα εμφυτευμάτων $W2V$ θα έχουν διάσταση $m$, ανάλογα το μοντέλο. 

Για κάθε ταινία d, μπορούμε να δημιουργήσουμε μια διανυσματική αναπαράσταση $W2V(d)$ διαστάσεων $m$ χρησιμοποιώντας το $tfidf(w_i)$ ως συντελεστή βαρύτητας για κάθε εμφύτευμα $W2V(w_i)$:

$$ W2V(d) = \frac{tfidf(w_1)\cdot W2V(w_i) + tfidf(w_2)\cdot W2V(w_2) + \dotsc  + tfidf(w_{N_{d}})\cdot W2V(w_{N_{d}})}{tfidf(w_1)+tfidf(w_2)+ \dotsc + tfidf(w_{N_{d}})}$$


#### build_tfw2v

Υλοποιήστε μια συνάρτηση `build_tfw2v` με ορίσματα:
- `corpus` που θα είναι το προεπεξεργασμένο dataset σας,
- `vectors` που θα είναι το μοντέλο που θα σας δίνει τα διανύσματα των εμφυτεύσεων vectors, και 
- `embeddings_size` που θα είναι η διάσταση των εμφυτευμάτων $m$.

H συνάρτηση αυτή θα επιστρέφει ένα νέο corpus που θα είναι ένας πίνακας 5000 (όσες οι ταινίες σας) x $m$ (το η διάσταση των εμφυτευμάτων). Ανάλογα ποιο μοντέλο χρησιμποιείτε για transfer learning ο πίνακας αυτός θα είναι διαφορετικός.

Μπορείτε πλεόν να καλείτε την `content_recommender` με διαφορετικά corpora στο όρισμα `corpus_type`. Σημειώστε ότι στο TFidfVectorizer χρησιμοποιουμε τη σειριακή μορφή των numpy arrays και ίσως σας χρησιμεύσει η `sparse.csr_matrix()` από την Scipy.

In [28]:
import math

def build_tfw2v(corpus,vectors,embeddings_size):
  new_corpus = []
  vectorizer = TfidfVectorizer()
  vectorizer.fit(corpus)
  corpus_tf_idf = vectorizer.transform(corpus).toarray()
  for i in range(0,len(corpus)):
    nom = 0
    den = 0
    words = word_tokenize(corpus[i])
    for j in range(0,len(words)):
      if words[j] in vectors:
        nom = nom + np.multiply(corpus_tf_idf[i][j],vectors[words[j]])
        den = den + corpus_tf_idf[i][j]
    if den != 0:
      new_corpus.append(np.divide(nom,den))
  return new_corpus

Χτίζουμε τώρα τα καινούργια corpora: 

In [29]:
twitter_corp = build_tfw2v(corpus,model1,100)

In [32]:
wiki_corp = build_tfw2v(corpus,model2,100)

Επαναλαμβάνουμε τα τεστ μας και για τα δύο νέα corpora: 

### **Test 1**

In [33]:
content_recommender(1,3,twitter_corp)

Target movie ID: 1
Target movie title: ['A Prize of Arms']
Target movie description: three criminal hatched plan army barrack troop dispatched take part war middle east believed large amount pay premise shipped gang enters army barrack disguised soldier proceeds pay corp headquarters guise maintenance work make sure alarm disabled give time make escape robbery take place rest day try integrate working base including vaccinated overseas service avoid attracting attention night fall change military police uniform head pay headquarters announcing arrival report fire begin searching room starting small blaze order premise evacuated building empty break safe steal £100,000 starting several fire cover activity withdraw carrying fake casualty stretcher troop rush across base put fire men drive secluded spot base left army truck officer ring medic check progress casualty told nobody arrived suspicious raise alarm whole camp put standby police sent initially fooled thinking criminal already lef

In [34]:
content_recommender(1,3,wiki_corp)

Target movie ID: 1
Target movie title: ['A Prize of Arms']
Target movie description: three criminal hatched plan army barrack troop dispatched take part war middle east believed large amount pay premise shipped gang enters army barrack disguised soldier proceeds pay corp headquarters guise maintenance work make sure alarm disabled give time make escape robbery take place rest day try integrate working base including vaccinated overseas service avoid attracting attention night fall change military police uniform head pay headquarters announcing arrival report fire begin searching room starting small blaze order premise evacuated building empty break safe steal £100,000 starting several fire cover activity withdraw carrying fake casualty stretcher troop rush across base put fire men drive secluded spot base left army truck officer ring medic check progress casualty told nobody arrived suspicious raise alarm whole camp put standby police sent initially fooled thinking criminal already lef

Παρατηρούμε πως για το corpus με γνώση από το twitter (twitter_corp), ο recommender δίνει τώρα καλές συστάσεις (και οι τρεις ταινίες έχουν σχέση με την τανία στόχο: οι δύο πρώτες ως σχετικές με το έγγλημα και η τρίτη ως δράμα). Για το corpus με γνώση από την wikipedia, οι συστάσεις δεν είναι καλές. 

### **Test 2**

In [35]:
content_recommender(10,3,twitter_corp)

Target movie ID: 10
Target movie title: ['Rabbit Transit']
Target movie description: relaxing steam bath bug read original fable reading credit tortoise beat hare becomes incensed idea turtle outrunning rabbit also steam bath claim could outrun bug prompting bug challenge race time bug agree cheating however quickly reveals rocket propelled allowing surprising combination fast slow bug best steal dismantle destroy device little effect end however bug manage top turtle cross finish line first nevertheless last laugh rook rabbit confessing 100 easy —in speed zone bug taken away police enjoy victory—behind bar close cartoon famously saying stinker
Target movie categories: ['"Short Film",  "Comedy film",  "Family Film",  "Animation"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 0.9989717988064513
Recommendation movie ID: 516
Recommendation movie title: ['Alien Thunder']
Recommendation movie description: based true follows canadian mountie want give indian fair trial murder fellow officer wind

In [36]:
content_recommender(10,3,wiki_corp)

Target movie ID: 10
Target movie title: ['Rabbit Transit']
Target movie description: relaxing steam bath bug read original fable reading credit tortoise beat hare becomes incensed idea turtle outrunning rabbit also steam bath claim could outrun bug prompting bug challenge race time bug agree cheating however quickly reveals rocket propelled allowing surprising combination fast slow bug best steal dismantle destroy device little effect end however bug manage top turtle cross finish line first nevertheless last laugh rook rabbit confessing 100 easy —in speed zone bug taken away police enjoy victory—behind bar close cartoon famously saying stinker
Target movie categories: ['"Short Film",  "Comedy film",  "Family Film",  "Animation"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.2118422836065292
Recommendation movie ID: 605
Recommendation movie title: ['Down and Outing']
Recommendation movie description: accompanies owner clobber severe anger management issue fishing seeing perfect opportuni

Εδώ παρατηρούμε πως για το twitter_corp έχουμε δύο σχετικά καλές συστάσεις (η 2η και η 3η), καθώς η μία ανήκει στο είδος family movie και η άλλη είναι ταινία με υπερήρωες, επομένως και οι δύο μπορεί να ενδιέφεραν κάποιον που είδε την ταινία στόχο (πιθανώς ένα παιδί). Για το wiki_corp η 1η σύσταση είναι επίσης καλή (ως κομωδία), αλλά οι άλλες δύο όχι. 

### **Test 3**

In [37]:
content_recommender(30,6,twitter_corp)

Target movie ID: 30
Target movie title: ['Cover Story']
Target movie description: khan tabu next door neighbour chandrashekar retired judge haunted past two become friend computer engineer wear high power contact lens vision problem meet news reporter executive director true vision chandrashekar killed mysterious man seen recognised wearing contact lens time corrupt policeman think killer eventually discovers actually killer also killed chandashekar search real killer police attempting catch form rest
Target movie categories: ['"Thriller",  "Crime Fiction",  "World cinema",  "Drama",  "Romantic drama",  "Crime Thriller",  "Romance Film"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.0884799286723137
Recommendation movie ID: 608
Recommendation movie title: ['Paranormal Activity 4']
Recommendation movie description: beginning show ending paranormal activity taking cutscene appears stating whereabouts remain unknown 2011 film younger brother soccer match house show boyfriend treehouse garde

In [38]:
content_recommender(30,6,wiki_corp)

Target movie ID: 30
Target movie title: ['Cover Story']
Target movie description: khan tabu next door neighbour chandrashekar retired judge haunted past two become friend computer engineer wear high power contact lens vision problem meet news reporter executive director true vision chandrashekar killed mysterious man seen recognised wearing contact lens time corrupt policeman think killer eventually discovers actually killer also killed chandashekar search real killer police attempting catch form rest
Target movie categories: ['"Thriller",  "Crime Fiction",  "World cinema",  "Drama",  "Romantic drama",  "Crime Thriller",  "Romance Film"']
Σειρά σύστασης: 1
Ομοιότητα συνημιτόνου: 1.2706857323646545
Recommendation movie ID: 416
Recommendation movie title: ['Swiss Miss']
Recommendation movie description: mousetrap salesman hoping better business switzerland theory cheese switzerland mouse visiting one village cheese shop owner con ware bogus banknote forced work dishwasher nearby hotel or

Παρατηρούμε πως ο recommender βγάζει καλές συστστάσεις για το twitter_corp (εκτός από την 4η που είναι documentary). Και για το wiki_corp έχουμε αρκετές καλές συστάσεις (όλες εκτός από την 4η και την 6η συνδέονται με την ταινία στόχο με κάποιον τρόπο, επειδή περιλαμβάνουν είτε εγκλήματα είτε κάποια ρομαντική πλοκή). 

### **Test 4**

In [41]:
content_recommender(300,5,twitter_corp)

Target movie ID: 300
Target movie title: ['A Bittersweet Life']
Target movie description: sun-woo high ranking mobster enforcer kang cold calculating crime bos unquestionably loyal two share concern business tension baek jr. rival family kang assigns sun-woo perceived simple errand away business young mistress heesoo fear affair another man giving sun-woo mandate kill manages discover performs duty following heesoo escorting music recital one day becomes quietly enthralled girl beauty innocence glimpse lonely empty personal life become prevalent come discover heesoo lover directly home fiercely beat prepares inform kang attraction cause hesitate thus spare two condition longer earning heesoo enmity meanwhile sun-woo continues embroiled personal business baek jr. beaten several henchman earlier overstaying welcome hotel threatened one enforcer apologize adamantly refuse fueled frustration heesoo relaxes apartment later one night suddenly kidnapped baek men tortured receive new order via

In [42]:
content_recommender(300,5,wiki_corp)

Target movie ID: 300
Target movie title: ['A Bittersweet Life']
Target movie description: sun-woo high ranking mobster enforcer kang cold calculating crime bos unquestionably loyal two share concern business tension baek jr. rival family kang assigns sun-woo perceived simple errand away business young mistress heesoo fear affair another man giving sun-woo mandate kill manages discover performs duty following heesoo escorting music recital one day becomes quietly enthralled girl beauty innocence glimpse lonely empty personal life become prevalent come discover heesoo lover directly home fiercely beat prepares inform kang attraction cause hesitate thus spare two condition longer earning heesoo enmity meanwhile sun-woo continues embroiled personal business baek jr. beaten several henchman earlier overstaying welcome hotel threatened one enforcer apologize adamantly refuse fueled frustration heesoo relaxes apartment later one night suddenly kidnapped baek men tortured receive new order via

Σε αυτήν την περίπτωση οι συστάσεις δεν είναι ιδιαίτερα καλές. Για το twitter_corp η 5η σύσταση έχει κάποια συνάφεια με την ταινία στόχο γιατί περιέχει εγκλήματα, αλλά οι υπόλοιπες είναι κακές. Για το wiki_corp είναι όλες κακές. 

**Συνολικά** Βλέπουμε καθαρά, λοιπόν, πως το σύστημα συστάσεών μας έχει συνολικά βελτιωθεί χρησιμοποιώντας τα embeddings. Η βελτίωση είναι πολύ πιο μεγάλη όταν χρησιμοποιούμε αυτό με γνώση από το twitter, όμως και αυτό με γνώση από την wikipedia επιφέρει κάποια βελτιώση.

## Ανάλυση αποτελεσμάτων

### Σύστημα συστάσεων βασισμένο μόνο στο tfidf

- Σε markdown περιγράψτε τι προεπεξεργασία κάνετε στα κείμενα και γιατί.

- Περιγράψτε πως προχωρήσατε στις επιλογές σας για τη βελτιστοποίηση της `TfidfVectorizer`. 

- [Cherry-picking:](https://www.wikiwand.com/en/Cherry_picking) Δώσετε παραδείγματα (IDs) από τη συλλογή σας που επιστρέφουν καλά αποτελέσματα μέχρι `max_recommendations` (τουλάχιστον 5) και σχολιάστε.

- [Nit-picking:](https://en.wikipedia.org/wiki/Nitpicking) Δώστε παραδείγματα (IDs) από τη συλλογή σας που επιστρέφουν κακά αποτελέσματα και σχολιάστε.

- Ποια είναι συνολικά τα πλεονεκτήματα και μειονεκτήματα ενός recommender βασισμένου στο tfidf;

### Σύγκριση και σχολιασμός με recommenders βασισμένων στο Word2Vec

- Υλoποιήστε recommenders που βασίζονται σε μεταφορά μάθησης και εμφυτεύματα. Χρησιμοποιήστε παραδείγματα για να υποδείξετε δυνατά και αδύναμα σημεία τους.

- Μπορείτε να σχολιάσετε τα recommenders που βασίζονται στο Word2Vec σε σχέση με το απλό μοντέλο tfidf, εξετάζοντας τις συστάσεις για ίδια ID.

- Μπορείτε επίσης να εξετάσετε συγκριτικά τα Word2Vec recommenders μεταξύ τους και πάλι βασιζόμενοι σε παραδείγματα.

- Οι παρατηρήσεις σας θα βασίζονται στην ανάλυση των ποιοτικών χαρακτηριστικών που είναι η σειρά και το σύνολο των συστάσεων. Ωστόσο, μπορείτε να συμπεριλάβετε και ποσοτικά χαρακτηριστικά όπως τους χρονους loading και συγκρότησης του corpus αλλά και της διαστατικότητας $m$.

Χρησιμοποιήστε όποια μορφή reporting κρίνετε καταλληλότερη: κείμενο, πίνακες, διαγράμματα.


**Σημείωση:** Η ανάλυση των αποτελεσμάτων έγινε σταδιακά σε κάθε βήμα. 

## Πρακτικό tip - persistence αντικειμένων με joblib.dump

Καθώς στην δεύτερη εργασία καλείστε να δημιουργήσετε διάφορα corpora των οποίων η δημιουργία παίρνει χρόνο, υπάρχει ένας εύκολος τρόπος να αποθηκεύουμε μεταβλητές σε dump files και να τις διαβάζουμε απευθείας.

H βιβλιοθήκη [joblib](https://pypi.python.org/pypi/joblib) της Python δίνει κάποιες εξαιρετικά χρήσιμες ιδιότητες στην ανάπτυξη κώδικα: pipelining, παραλληλισμό, caching και variable persistence. Τις τρεις πρώτες ιδιότητες τις είδαμε στην πρώτη άσκηση. Στην παρούσα άσκηση θα μας φανεί χρήσιμη η τέταρτη, το persistence των αντικειμένων. Συγκεκριμένα μπορούμε με:

```python
joblib.dump(my_object, 'my_object.pkl') 
```

να αποθηκεύσουμε οποιοδήποτε αντικείμενο-μεταβλητή (εδώ το `my_object`) απευθείας πάνω στο filesystem ως αρχείο, το οποίο στη συνέχεια μπορούμε να ανακαλέσουμε ως εξής:

```python
my_object = joblib.load('my_object.pkl')
```

Μπορούμε έτσι να ανακαλέσουμε μεταβλητές ακόμα και αφού κλείσουμε και ξανανοίξουμε το notebook, χωρίς να χρειαστεί να ακολουθήσουμε ξανά όλα τα βήματα ένα - ένα για την παραγωγή τους, κάτι ιδιαίτερα χρήσιμο αν αυτή η διαδικασία είναι χρονοβόρα.

Ας αποθηκεύσουμε το `corpus_tf_idf` και στη συνέχεια ας το ανακαλέσουμε.

In [ ]:
import joblib

joblib.dump(corpus_tf_idf_plain, 'corpus_tf_idf.pkl') 



Μπορείτε με ένα απλό `!ls` να δείτε ότι το αρχείο `corpus_tf_idf.pkl` υπάρχει στο filesystem σας (== persistence):

In [ ]:
!ls -lh

και μπορούμε να τα διαβάσουμε με `joblib.load`

In [ ]:
corpus_tf_idf = joblib.load('corpus_tf_idf.pkl')

# Εφαρμογή 2.  Τοπολογική και σημασιολογική απεικόνιση της ταινιών με χρήση SOM
<img src="https://i.imgur.com/Z4FdurD.jpg" width="60%">

## Δημιουργία dataset
Στη δεύτερη εφαρμογή θα βασιστούμε στις τοπολογικές ιδιότητες των Self Organizing Maps (SOM) για να φτιάξουμε ενά χάρτη (grid) δύο διαστάσεων όπου θα απεικονίζονται όλες οι ταινίες της συλλογής της ομάδας με τρόπο χωρικά συνεκτικό ως προς το περιεχόμενο και κυρίως το είδος τους (ο παραπάνω χάρτης είναι ενδεικτικός, δεν αντιστοιχεί στο dataset μας). 

Διαλέξτε για την αναπαράσταση των documents αυτήν που πιστεύετε απέδωσε καλύτερα στο πρώτα σκέλος της άσκησης. Έστω ότι αυτή είναι η `my_best_corpus`.

Η έτοιμη συνάρτηση `build_final_set` θα ενώσει την αναπαράσταση που θα της δώσετε ως όρισμα `mycorpus` με τις binarized κατηγορίες `catbins` των ταινιών ως επιπλέον κολόνες (χαρακτηριστικά). Συνεπώς, κάθε ταινία αναπαρίσταται στο Vector Space Model από τα χαρακτηριστικά της αναπαράστασης `mycorpus` και τις κατηγορίες της.

Τέλος, η συνάρτηση δέχεται ένα ορισμα για το πόσες ταινίες να επιστρέψει, με default τιμή όλες τις ταινίες (5000). Αυτό είναι χρήσιμο για να μπορείτε αν θέλετε να φτιάχνετε μικρότερα σύνολα δεδομένων ώστε να εκπαιδεύεται ταχύτερα το SOM. 

Θα τρέχουμε τη συνάρτηση με `final_set = build_final_set(my_best_corpus)`.

In [ ]:
def build_final_set(mycorpus, doc_limit = 5000, tf_idf_only=False):
    # convert sparse tf_idf to dense tf_idf representation
    dense_tf_idf = mycorpus.toarray()[0:doc_limit,:]
    if tf_idf_only:
        # use only tf_idf
        final_set = dense_tf_idf
    else:
        # append the binary categories features horizontaly to the (dense) tf_idf features
        final_set = np.hstack((dense_tf_idf, catbins[0:doc_limit,:]))
    # η somoclu θέλει δεδομ΄ένα σε float32
    return np.array(final_set, dtype=np.float32)

Στο επόμενο κελί, τυπώνουμε τις διαστάσεις του τελικού dataset μας. **Χωρίς βελτιστοποίηση του TFIDF** θα έχουμε περίπου 50.000 χαρακτηριστικά και ο θα είναι ανέφικτο να προχωρήσουμε στην εκπαίδευση του SOM.

In [ ]:
final_set.shape

## Εκπαίδευση χάρτη SOM

Θα δουλέψουμε με τη βιβλιοθήκη SOM ["Somoclu"](http://somoclu.readthedocs.io/en/stable/index.html). Εισάγουμε τις somoclu και matplotlib και λέμε στη matplotlib να τυπώνει εντός του notebook (κι όχι σε pop up window).

In [ ]:
# install somoclu
!pip install --upgrade somoclu
# import sompoclu, matplotlib
import somoclu
import matplotlib
# we will plot inside the notebook and not in separate window
%matplotlib inline

Καταρχάς διαβάστε το [function reference](http://somoclu.readthedocs.io/en/stable/reference.html) του somoclu. Θα δoυλέψουμε με χάρτη τύπου planar, παραλληλόγραμμου σχήματος νευρώνων με τυχαία αρχικοποίηση (όλα αυτά είναι default). Μπορείτε να δοκιμάσετε διάφορα μεγέθη χάρτη ωστόσο όσο ο αριθμός των νευρώνων μεγαλώνει, μεγαλώνει και ο χρόνος εκπαίδευσης. Για το training δεν χρειάζεται να ξεπεράσετε τα 100 epochs. Σε γενικές γραμμές μπορούμε να βασιστούμε στις default παραμέτρους μέχρι να έχουμε τη δυνατότητα να οπτικοποιήσουμε και να αναλύσουμε ποιοτικά τα αποτελέσματα. Ξεκινήστε με ένα χάρτη 10 x 10, 100 epochs training και ένα υποσύνολο των ταινιών (π.χ. 2000). Χρησιμοποιήστε την `time` για να έχετε μια εικόνα των χρόνων εκπαίδευσης. 


## Best matching units

Μετά από κάθε εκπαίδευση αποθηκεύστε σε μια μεταβλητή τα best matching units (bmus) για κάθε ταινία. Τα bmus μας δείχνουν σε ποιο νευρώνα ανήκει η κάθε ταινία. **Προσοχή: η σύμβαση των συντεταγμένων των νευρώνων στη Somoclu είναι (στήλη, γραμμή) δηλαδή το ανάποδο από την Python**. Με χρήση της [np.unique](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.unique.html) (μια πολύ χρήσιμη συνάρτηση στην άσκηση) αποθηκεύστε τα μοναδικά best matching units και τους δείκτες τους (indices) προς τις ταινίες. 

Σημειώστε ότι μπορεί να έχετε λιγότερα μοναδικά bmus από αριθμό νευρώνων γιατί μπορεί σε κάποιους νευρώνες να μην έχουν ανατεθεί ταινίες. Ως αριθμό νευρώνα θα θεωρήσουμε τον αριθμό γραμμής στον πίνακα μοναδικών bmus.



## Ομαδοποίηση (clustering)

Τυπικά, η ομαδοποίηση σε ένα χάρτη SOM προκύπτει από το unified distance matrix (U-matrix): για κάθε κόμβο υπολογίζεται η μέση απόστασή του από τους γειτονικούς κόμβους. Εάν χρησιμοποιηθεί μπλε χρώμα στις περιοχές του χάρτη όπου η τιμή αυτή είναι χαμηλή (μικρή απόσταση) και κόκκινο εκεί που η τιμή είναι υψηλή (μεγάλη απόσταση), τότε μπορούμε να πούμε ότι οι μπλε περιοχές αποτελούν clusters και οι κόκκινες αποτελούν σύνορα μεταξύ clusters.

To somoclu δίνει την επιπρόσθετη δυνατότητα να κάνουμε ομαδοποίηση των νευρώνων χρησιμοποιώντας οποιονδήποτε αλγόριθμο ομαδοποίησης του scikit-learn. Στην άσκηση θα χρησιμοποιήσουμε τον k-Means. Για τον αρχικό σας χάρτη δοκιμάστε ένα k=20 ή 25. Οι δύο προσεγγίσεις ομαδοποίησης είναι διαφορετικές, οπότε περιμένουμε τα αποτελέσματα να είναι κοντά αλλά όχι τα ίδια.



## Αποθήκευση του SOM

Επειδή η αρχικοποίηση του SOM γίνεται τυχαία και το clustering είναι και αυτό στοχαστική διαδικασία, οι θέσεις και οι ετικέτες των νευρώνων και των clusters θα είναι διαφορετικές κάθε φορά που τρέχετε τον χάρτη, ακόμα και με τις ίδιες παραμέτρους. Για να αποθηκεύσετε ένα συγκεκριμένο som και clustering χρησιμοποιήστε και πάλι την `joblib`. Μετά την ανάκληση ενός SOM θυμηθείτε να ακολουθήσετε τη διαδικασία για τα bmus.



## Οπτικοποίηση U-matrix, clustering και μέγεθος clusters

Για την εκτύπωση του U-matrix χρησιμοποιήστε τη `view_umatrix` με ορίσματα `bestmatches=True` και `figsize=(15, 15)` ή `figsize=(20, 20)`. Τα διαφορετικά χρώματα που εμφανίζονται στους κόμβους αντιπροσωπεύουν τα διαφορετικά clusters που προκύπτουν από τον k-Means. Μπορείτε να εμφανίσετε τη λεζάντα του U-matrix με το όρισμα `colorbar`. Μην τυπώνετε τις ετικέτες (labels) των δειγμάτων, είναι πολύ μεγάλος ο αριθμός τους.

Για μια δεύτερη πιο ξεκάθαρη οπτικοποίηση του clustering τυπώστε απευθείας τη μεταβλητή `clusters`.

Τέλος, χρησιμοποιώντας πάλι την `np.unique` (με διαφορετικό όρισμα) και την `np.argsort` (υπάρχουν και άλλοι τρόποι υλοποίησης) εκτυπώστε τις ετικέτες των clusters (αριθμοί από 0 έως k-1) και τον αριθμό των νευρώνων σε κάθε cluster, με φθίνουσα ή αύξουσα σειρά ως προς τον αριθμό των νευρώνων. Ουσιαστικά είναι ένα εργαλείο για να βρίσκετε εύκολα τα μεγάλα και μικρά clusters. 

Ακολουθεί ένα μη βελτιστοποιημένο παράδειγμα για τις τρεις προηγούμενες εξόδους:

<img src="https://image.ibb.co/i0tsfR/umatrix_s.jpg" width="35%">
<img src="https://image.ibb.co/nLgHEm/clusters.png" width="35%">




## Σημασιολογική ερμηνεία των clusters

Προκειμένου να μελετήσουμε τις τοπολογικές ιδιότητες του SOM και το αν έχουν ενσωματώσει σημασιολογική πληροφορία για τις ταινίες διαμέσου της διανυσματικής αναπαράστασης του tf-idf, των εμφυτευμάτων και των κατηγοριών, χρειαζόμαστε ένα κριτήριο ποιοτικής επισκόπησης των clusters. 

Θα υλοποιήσουμε το εξής κριτήριο: Λαμβάνουμε όρισμα έναν αριθμό (ετικέτα) cluster. Για το cluster αυτό βρίσκουμε όλους τους νευρώνες που του έχουν ανατεθεί από τον k-Means. Για όλους τους νευρώνες αυτούς βρίσκουμε όλες τις ταινίες που τους έχουν ανατεθεί (για τις οποίες αποτελούν bmus). Για όλες αυτές τις ταινίες τυπώνουμε ταξινομημένη τη συνολική στατιστική όλων των ειδών (κατηγοριών) και τις συχνότητές τους. Αν το cluster διαθέτει καλή συνοχή και εξειδίκευση, θα πρέπει κάποιες κατηγορίες να έχουν σαφώς μεγαλύτερη συχνότητα από τις υπόλοιπες. Θα μπορούμε τότε να αναθέσουμε αυτήν/ές την/τις κατηγορία/ες ως ετικέτες κινηματογραφικού είδους στο cluster.

Μπορείτε να υλοποιήσετε τη συνάρτηση αυτή όπως θέλετε. Μια πιθανή διαδικασία θα μπορούσε να είναι η ακόλουθη:

1. Ορίζουμε συνάρτηση `print_categories_stats` που δέχεται ως είσοδο λίστα με ids ταινιών. Δημιουργούμε μια κενή λίστα συνολικών κατηγοριών. Στη συνέχεια, για κάθε ταινία επεξεργαζόμαστε το string `categories` ως εξής: δημιουργούμε μια λίστα διαχωρίζοντας το string κατάλληλα με την `split` και αφαιρούμε τα whitespaces μεταξύ ετικετών με την `strip`. Προσθέτουμε τη λίστα αυτή στη συνολική λίστα κατηγοριών με την `extend`. Τέλος χρησιμοποιούμε πάλι την `np.unique` για να μετρήσουμε συχνότητα μοναδικών ετικετών κατηγοριών και ταξινομούμε με την `np.argsort`. Τυπώνουμε τις κατηγορίες και τις συχνότητες εμφάνισης ταξινομημένα. Χρήσιμες μπορεί να σας φανούν και οι `np.ravel`, `np.nditer`, `np.array2string` και `zip`.

2. Ορίζουμε τη βασική μας συνάρτηση `print_cluster_neurons_movies_report` που δέχεται ως όρισμα τον αριθμό ενός cluster. Με τη χρήση της `np.where` μπορούμε να βρούμε τις συντεταγμένες των bmus που αντιστοιχούν στο cluster και με την `column_stack` να φτιάξουμε έναν πίνακα bmus για το cluster. Προσοχή στη σειρά (στήλη - σειρά) στον πίνακα bmus. Για κάθε bmu αυτού του πίνακα ελέγχουμε αν υπάρχει στον πίνακα μοναδικών bmus που έχουμε υπολογίσει στην αρχή συνολικά και αν ναι προσθέτουμε το αντίστοιχο index του νευρώνα σε μια λίστα. Χρήσιμες μπορεί να είναι και οι `np.rollaxis`, `np.append`, `np.asscalar`. Επίσης πιθανώς να πρέπει να υλοποιήσετε ένα κριτήριο ομοιότητας μεταξύ ενός bmu και ενός μοναδικού bmu από τον αρχικό πίνακα bmus.

3. Υλοποιούμε μια βοηθητική συνάρτηση `neuron_movies_report`. Λαμβάνει ένα σύνολο νευρώνων από την `print_cluster_neurons_movies_report` και μέσω της `indices` φτιάχνει μια λίστα με το σύνολο ταινιών που ανήκουν σε αυτούς τους νευρώνες. Στο τέλος καλεί με αυτή τη λίστα την `print_categories_stats` που τυπώνει τις στατιστικές των κατηγοριών.

Μπορείτε βέβαια να προσθέσετε οποιαδήποτε επιπλέον έξοδο σας βοηθάει. Μια χρήσιμη έξοδος είναι πόσοι νευρώνες ανήκουν στο cluster και σε πόσους και ποιους από αυτούς έχουν ανατεθεί ταινίες.

Θα επιτελούμε τη σημασιολογική ερμηνεία του χάρτη καλώντας την `print_cluster_neurons_movies_report` με τον αριθμός ενός cluster που μας ενδιαφέρει. 

Παράδειγμα εξόδου για ένα cluster (μη βελτιστοποιημένος χάρτης, ωστόσο βλέπετε ότι οι μεγάλες κατηγορίες έχουν σημασιολογική  συνάφεια):

```
Overall Cluster Genres stats:  
[('"Horror"', 86), ('"Science Fiction"', 24), ('"B-movie"', 16), ('"Monster movie"', 10), ('"Creature Film"', 10), ('"Indie"', 9), ('"Zombie Film"', 9), ('"Slasher"', 8), ('"World cinema"', 8), ('"Sci-Fi Horror"', 7), ('"Natural horror films"', 6), ('"Supernatural"', 6), ('"Thriller"', 6), ('"Cult"', 5), ('"Black-and-white"', 5), ('"Japanese Movies"', 4), ('"Short Film"', 3), ('"Drama"', 3), ('"Psychological thriller"', 3), ('"Crime Fiction"', 3), ('"Monster"', 3), ('"Comedy"', 2), ('"Western"', 2), ('"Horror Comedy"', 2), ('"Archaeology"', 2), ('"Alien Film"', 2), ('"Teen"', 2), ('"Mystery"', 2), ('"Adventure"', 2), ('"Comedy film"', 2), ('"Combat Films"', 1), ('"Chinese Movies"', 1), ('"Action/Adventure"', 1), ('"Gothic Film"', 1), ('"Costume drama"', 1), ('"Disaster"', 1), ('"Docudrama"', 1), ('"Film adaptation"', 1), ('"Film noir"', 1), ('"Parody"', 1), ('"Period piece"', 1), ('"Action"', 1)]```
   


## Tips για το SOM και το clustering

- Για την ομαδοποίηση ένα U-matrix καλό είναι να εμφανίζει και μπλε-πράσινες περιοχές (clusters) και κόκκινες περιοχές (ορίων). Παρατηρήστε ποια σχέση υπάρχει μεταξύ αριθμού ταινιών στο final set, μεγέθους grid και ποιότητας U-matrix.
- Για το k του k-Means προσπαθήστε να προσεγγίζει σχετικά τα clusters του U-matrix (όπως είπαμε είναι διαφορετικοί μέθοδοι clustering). Μικρός αριθμός k δεν θα σέβεται τα όρια. Μεγάλος αριθμός θα δημιουργεί υπο-clusters εντός των clusters που φαίνονται στο U-matrix. Το τελευταίο δεν είναι απαραίτητα κακό, αλλά μεγαλώνει τον αριθμό clusters που πρέπει να αναλυθούν σημασιολογικά.
- Σε μικρούς χάρτες και με μικρά final sets δοκιμάστε διαφορετικές παραμέτρους για την εκπαίδευση του SOM. Σημειώστε τυχόν παραμέτρους που επηρεάζουν την ποιότητα του clustering για το dataset σας ώστε να τις εφαρμόσετε στους μεγάλους χάρτες.
- Κάποια τοπολογικά χαρακτηριστικά εμφανίζονται ήδη σε μικρούς χάρτες. Κάποια άλλα χρειάζονται μεγαλύτερους χάρτες. Δοκιμάστε μεγέθη 20x20, 25x25 ή και 30x30 και αντίστοιχη προσαρμογή των k. Όσο μεγαλώνουν οι χάρτες, μεγαλώνει η ανάλυση του χάρτη αλλά μεγαλώνει και ο αριθμός clusters που πρέπει να αναλυθούν.




## Ανάλυση τοπολογικών ιδιοτήτων χάρτη SOM

Μετά το πέρας της εκπαίδευσης και του clustering θα έχετε ένα χάρτη με τοπολογικές ιδιότητες ως προς τα είδη των ταίνιών της συλλογής σας, κάτι αντίστοιχο με την εικόνα στην αρχή της Εφαρμογής 2 αυτού του notebook. Η συγκεκριμένη εικόνα είναι μόνο για εικονογράφιση, δεν είναι χάρτης SOM καιδεν έχει καμία σχέση με τη συλλογή δεδομένων και τις κατηγορίες μας.

Για τον τελικό χάρτη SOM που θα παράξετε για τη συλλογή σας, αναλύστε σε markdown με συγκεκριμένη αναφορά σε αριθμούς clusters και τη σημασιολογική ερμηνεία τους τις εξής τρεις τοπολογικές ιδιότητες του SOM: 

1. Δεδομένα που έχουν μεγαλύτερη πυκνότητα πιθανότητας στο χώρο εισόδου τείνουν να απεικονίζονται με περισσότερους νευρώνες στο χώρο μειωμένης διαστατικότητας. Δώστε παραδείγματα από συχνές και λιγότερο συχνές κατηγορίες ταινιών. Χρησιμοποιήστε τις στατιστικές των κατηγοριών στη συλλογή σας και τον αριθμό κόμβων που χαρακτηρίζουν.
2. Μακρινά πρότυπα εισόδου τείνουν να απεικονίζονται απομακρυσμένα στο χάρτη. Υπάρχουν χαρακτηριστικές κατηγορίες ταινιών που ήδη από μικρούς χάρτες τείνουν να τοποθετούνται σε διαφορετικά ή απομονωμένα σημεία του χάρτη.
3. Κοντινά πρότυπα εισόδου τείνουν να απεικονίζονται κοντά στο χάρτη. Σε μεγάλους χάρτες εντοπίστε είδη ταινιών και κοντινά τους υποείδη.

Προφανώς τοποθέτηση σε 2 διαστάσεις που να σέβεται μια απόλυτη τοπολογία δεν είναι εφικτή, αφενός γιατί δεν υπάρχει κάποια απόλυτη εξ ορισμού για τα κινηματογραφικά είδη ακόμα και σε πολλές διαστάσεις, αφετέρου γιατί πραγματοποιούμε μείωση διαστατικότητας.

Εντοπίστε μεγάλα clusters και μικρά clusters που δεν έχουν σαφή χαρακτηριστικά. Εντοπίστε clusters συγκεκριμένων ειδών που μοιάζουν να μην έχουν τοπολογική συνάφεια με γύρω περιοχές. Προτείνετε πιθανές ερμηνείες.


Τέλος, εντοπίστε clusters που έχουν κατά την άποψή σας ιδιαίτερο ενδιαφέρον στη συλλογή της ομάδας σας (data exploration / discovery value) και σχολιάστε.



# Τελική παράδοση άσκησης

- Θα παραδώσετε στο helios το παρόν notebook επεξεργασμένο ή ένα ή δύο νέα zipαρισμένα με τις απαντήσεις σας για τα ζητούμενα και των δύο εφαρμογών. 
- Θυμηθείτε ότι η ανάλυση του χάρτη στο markdown με αναφορά σε αριθμούς clusters πρέπει να αναφέρεται στον τελικό χάρτη με τα κελιά ορατά που θα παραδώσετε αλλιώς ο χάρτης που θα προκύψει θα είναι διαφορετικός και τα labels των clusters δεν θα αντιστοιχούν στην ανάλυσή σας. 
- Μην ξεχάσετε στην αρχή ένα κελί markdown με **τα στοιχεία της ομάδας σας**.

<table>
  <tr><td align="center">
    <font size="4">Παρακαλούμε διατρέξτε βήμα-βήμα το notebook για να μην ξεχάσετε παραδοτέα</font>
</td>
  </tr>
</table>